# 🎬 Модуль 0. Введение: что такое API и окружение
### Курс: «Проектирование и разработка API»

**Сквозной кейс:** кинотеатр «Искорка»  
**Объём:** 4 академических часа  
**Артефакт модуля:** рабочее окружение + заметки по первым запросам

---

## 📖 О чём этот модуль

Мы начинаем курс с **выравнивания базы**. К концу модуля вы сможете:

- Объяснить, что такое **API** и **контракт**.
- Прочитать **HTTP-запрос** и **HTTP-ответ**.
- Отправить запрос и **разобрать JSON**.
- Понимать, **как устроен курс** и где вы находитесь на его карте.

> 💡 **Главная мысль модуля:** API — это **договор**. Договор между тем, кто **предоставляет** данные/функции, и тем, кто их **использует**. Наша задача в курсе — научиться **проектировать** такие договоры осознанно.

---

## 🎯 Проблема из кейса

Кинотеатр **«Искорка»** показывает фильмы для детей и подростков. У него есть:

- **Онлайн-расписание** — витрина для зрителей.
- **Система бронирования** — продажа билетов через сайт.
- **Административная панель** — управление сеансами.

Все данные хранятся и обрабатываются **на одном сервере**.

**Четыре задачи, которые нужно автоматизировать:**

| № | Задача | Кто пользователь |
|---|--------|------------------|
| 1 | Узнать наличие мест на сеанс | Зритель |
| 2 | Забронировать место | Зритель |
| 3 | Отменить бронь | Зритель |
| 4 | Изменить время сеанса | Администратор |

**Вопрос, который покажется странным, но важный:** почему сайт бронирования, касса кинотеатра и админ-панель должны работать с **одним и тем же API**, а не каждая со своей базой?

> 💡 Ответ вы найдёте в этом модуле. Он — в слове **«контракт»**.

In [4]:
# ============================================================
#  Проверка окружения: для этого модуля нужны requests и json
# ============================================================
import json
import sys

print('Python:', sys.version.split()[0])

try:
    import requests
    print('requests:', requests.__version__, '✅')
except ImportError:
    print('requests не установлен. Установите: pip install requests')

try:
    import yaml
    print('PyYAML:', yaml.__version__, '✅')
except ImportError:
    print('PyYAML не установлен. Установите: pip install pyyaml')

Python: 3.15.2
requests: 2.34.2 ✅
PyYAML не установлен. Установите: pip install pyyaml


In [ ]:
pip install -U pip setuptools wheel

In [ ]:
pip install --only-binary=yaml pyyaml

In [ ]:
pip install ruamel.yaml

---
# 1. Краткая история API: от системных вызовов до событийных архитектур
---

История API — это не просто хронология технологий. Это история о том, как программы учились **договариваться** друг с другом: сначала внутри одного компьютера, затем через сеть, а сегодня — через распределённые события. Понимание этой эволюции помогает осознанно выбирать стиль API для конкретной задачи, а не следовать моде.

## 📜 Период 1. 1950–1970-е: Монолитные приложения и системные вызовы

**Исторический контекст.** Компьютеры этого времени — это **мейнфреймы**, а программы — **монолитные, тесно связанные системы**. Взаимодействие между программами было минимальным и **аппаратно-зависимым**. Не существовало концепции «API» в современном смысле.

**Ключевые разработки.**
- **Системные вызовы (system calls)** — единственный способ, которым программа могла запросить у операционной системы выполнение привилегированных операций (работа с файлами, памятью, устройствами).
- **API на уровне языка** — наборы функций в библиотеках (например, стандартная библиотека C). Термин «Application Programming Interface» начал использоваться в 1970-х, в частности, в работе 1974 года, сравнивающей реляционный и сетевой подходы.

**Пример.** Программа на C вызывает `read()` для чтения файла. Это — «API» операционной системы, но он **не выходит за пределы одной машины** и жёстко привязан к конкретной ОС.

> 💡 **Значение для курса:** именно на этом этапе сформировалось базовое понимание **контракта** — «если ты вызовешь функцию с такими аргументами, она вернёт такой результат». Но этот контракт был **локальным** и **тесно связанным** с реализацией.

---

## 📜 Период 2. 1980–1990-е: Библиотеки, фреймворки и первые распределённые вызовы

**Исторический контекст.** Появились **персональные компьютеры** и **локальные сети**. Программы стали сложнее, возникла потребность в **переиспользовании кода** и **модульности**.

**Ключевые разработки.**
- **Библиотеки и фреймворки:** C Standard Library, POSIX API, Win32 API, Java API. API теперь определяли **функции и методы**, обеспечивая модульное программирование и абстракцию.
- **Remote Procedure Call (RPC):** идея **вызывать функцию на удалённой машине так, как если бы она была локальной**. RPC возник в 1960–1990-х. В 1984 году Биррелл и Нельсон формализовали механизм RPC.
- **Распределённые объектные технологии:** **CORBA** (начало разработки в конце 1980-х под эгидой OMG), **DCOM** (Microsoft) и **Java RMI**. DCE (Distributed Computing Environment) заложил основу для CORBA и DCOM.

**Пример.** Java-программа вызывает метод `getWeather()` на удалённом сервере через RMI. Для разработчика это выглядит как обычный вызов, но «под капотом» происходит сериализация, передача по сети и десериализация.

**Ограничения.** RPC был **тесно связанным** (tightly coupled): изменения на сервере часто требовали изменений на клиенте. Кроме того, RPC был **зависим от языка** и сложен в настройке.

> 💡 **Значение для курса:** RPC — прямой предок **gRPC**, который мы будем изучать в Модуле 6. Понимание его ограничений помогает увидеть, почему позже появились REST и событийные подходы.

---

## 📜 Период 3. 2000: SOAP, XML и рождение REST

**Исторический контекст.** Конец 1990-х — начало 2000-х — эпоха **SOA (Service-Oriented Architecture)**. Бизнес хотел интегрировать разнородные системы, и нужен был **формальный, строгий контракт**.

**Ключевые разработки.**
- **SOAP (Simple Object Access Protocol):** протокол на основе **XML** для обмена структурированными сообщениями. SOAP доминировал в **корпоративной интеграции** (банки, enterprise-сервисы) благодаря строгим контрактам и формальным стандартам.
- **REST (Representational State Transfer):** **Рой Филдинг** в **2000 году** в докторской диссертации в Калифорнийском университете (Ирвайн) сформулировал REST как **архитектурный стиль** для распределённых систем. REST делал акцент на **stateless-взаимодействии**, **ресурсах** и **стандартных HTTP-методах**.

**Пример.** SOAP-сервис для бронирования билетов описан в **WSDL** (Web Services Description Language). Клиент генерирует код из WSDL и вызывает методы. REST-сервис для той же задачи использует `POST /bookings` с JSON-телом.

> 💡 **Значение для курса:** противостояние SOAP и REST в начале 2000-х — это **первый большой спор о стилях API**. REST победил в вебе и мобильных приложениях благодаря **простоте и масштабируемости**. Именно REST станет основой Модулей 3–5.

---

## 📜 Период 4. 2006: API как продукт (Amazon Web Services)

**Исторический контекст.** Amazon, будучи интернет-магазином, накопил огромную инфраструктуру. Возникла идея **продавать доступ к ней** другим разработчикам.

**Ключевые разработки.**
- **Amazon S3** (март 2006) — первый крупный публичный продукт AWS, предоставляющий объектное хранилище через **RESTful API**.
- **Amazon EC2** (август 2006) — вычислительные мощности по запросу, также изначально работавшие **исключительно как RESTful API**.

**Пример.** Разработчик регистрируется в AWS, получает ключи доступа и через HTTP-запросы к `s3.amazonaws.com` создаёт бакеты и загружает файлы. **API становится продуктом**, который можно продавать.

> 💡 **Значение для курса:** с этого момента API — не просто «техническая прослойка», а **бизнес-актив**. В курсе мы будем проектировать API так, чтобы они были **удобны для потребителей** (документация, mock, версионирование).

---

## 📜 Период 5. 2010-е: JSON, OpenAPI, мобильные приложения и API-экономика

**Исторический контекст.** Взрывной рост **мобильных приложений** и **одностраничных веб-приложений (SPA)** потребовал лёгких и быстрых API. XML уступил место **JSON**.

**Ключевые разработки.**
- **JSON как доминирующий формат:** переход от XML к JSON для **лёгкого, читаемого и быстрого** обмена данными. Современные REST API используют JSON-полезные нагрузки.
- **OpenAPI / Swagger:** в **2010 году** Тони Тэм (Wordnik) начал разработку **Swagger** — стандартизированного описания API. В **2015 году** Swagger был передан **OpenAPI Initiative** и переименован в **OpenAPI Specification**. Это позволило **генерировать документацию, код и тесты** из единого контракта.
- **API-экономика и микросервисы:** API становятся **продуктами**, а микросервисная архитектура и cloud-native системы **строятся вокруг API**.

**Пример.** Мобильное приложение «Искорки» получает расписание через `GET /sessions`, а данные приходят в JSON. Контракт описан в `openapi.yaml`, из которого генерируется mock-сервер для параллельной разработки.

> 💡 **Значение для курса:** это **прямой мостик к нашему кейсу**. Мы будем использовать JSON, OpenAPI и Postman — инструменты, рождённые в эту эпоху.

---

## 📜 Период 6. 2015: gRPC и GraphQL — альтернативы REST

**Исторический контекст.** REST доминировал, но у него были ограничения: **over-fetching** (клиент получает больше данных, чем нужно) и **under-fetching** (требуется несколько запросов, чтобы собрать данные). Для микросервисов нужна была **высокая производительность**.

**Ключевые разработки.**
- **gRPC:** Google **открыл исходный код в 2015 году**. Это высокопроизводительный **RPC-фреймворк** на **HTTP/2** с использованием **Protocol Buffers** для строгой типизации и генерации кода. Идеален для **server-to-server** взаимодействия в микросервисах.
- **GraphQL:** Facebook разработал GraphQL с 2012 года и **опубликовал спецификацию в 2015 году**. Это **язык запросов** и серверный runtime, который позволяет клиенту **запросить ровно те данные, которые ему нужны**, через один эндпоинт.

**Пример.** Администратор «Искорки» переносит сеанс через gRPC-вызов `RescheduleSession`. Мобильное приложение запрашивает через GraphQL только `id` и `startTime` сеанса, не получая лишних полей.

> 💡 **Значение для курса:** в Модуле 6 мы сравним REST, gRPC и GraphQL на **одной и той же задаче**, чтобы студенты увидели **компромиссы** каждого стиля.

---

## 📜 Период 7. 2017–настоящее время: AsyncAPI и событийные архитектуры

**Исторический контекст.** Микросервисы и распределённые системы показали, что **синхронных вызовов недостаточно**. Возникла потребность в **асинхронном, событийном** взаимодействии: «произошло событие — отреагировали».

**Ключевые разработки.**
- **AsyncAPI:** инициатива началась в компании **Hitch**, где оценили преимущества OpenAPI, но **не могли применить его к архитектуре, ориентированной на сообщения**. Первая версия **1.0** вышла в **2017 году**, версия **2.0** — в **2019 году**. AsyncAPI описывает **каналы, сообщения и серверы** для событийных API, поддерживая **Kafka, RabbitMQ, WebSockets** и другие.
- **Event-Driven Architecture (EDA):** архитектурный стиль, в котором **события** являются первичным способом интеграции. Событие — это **факт** («бронь создана», «сеанс изменён»), который публикуется и потребляется асинхронно.

**Пример.** После создания брони «Искорка» публикует событие `BookingCreated` в Kafka. Платёжный сервис и сервис уведомлений **независимо** подписаны на этот топик и реагируют каждый по-своему.

> 💡 **Значение для курса:** это **финальный и самый современный** слой курса. В Модулях 7–9 студенты научатся проектировать асинхронные и событийные интеграции с помощью RabbitMQ, Kafka и AsyncAPI.

---

## 📊 Сводная таблица эволюции API

| Период | Этап | Ключевая идея | Примеры |
|--------|------|---------------|---------|
| **1950–1970-е** | Монолитные приложения | Программа сама себе API | Системные вызовы ОС |
| **1980–1990-е** | Библиотеки и RPC | Вызвать функцию удалённо | POSIX, CORBA, DCOM, Java RMI |
| **2000** | SOAP и REST | Строгий контракт vs простота HTTP | SOAP/WSDL, REST (Филдинг) |
| **2006** | API как продукт | Продавать доступ через API | Amazon S3, EC2 |
| **2010-е** | JSON и OpenAPI | Контракт как код, документация | Swagger/OpenAPI, JSON |
| **2015** | gRPC и GraphQL | Альтернативы REST для разных задач | gRPC (Google), GraphQL (Facebook) |
| **2017+** | AsyncAPI и EDA | События как первичный способ интеграции | AsyncAPI, Kafka, RabbitMQ |

> 🎯 **Главный вывод раздела:** история API — это движение от **тесной связи** к **слабой связанности**. Чем слабее связаны системы, тем легче их менять и масштабировать. Именно этот принцип ляжет в основу всех решений в курсе.

## 🧠 Что такое API (определение для курса)

**API (Application Programming Interface)** — это **контракт**, по которому одна программа может запрашивать данные или действия у другой.

Ключевые элементы контракта:

| Элемент | Что описывает | Пример для «Искорки» |
|---------|---------------|----------------------|
| **Ресурс / операция** | Что доступно снаружи | «Наличие мест на сеанс» |
| **Формат запроса** | Как вызвать | `GET /sessions/42/seats` |
| **Формат ответа** | Что вернётся | Список мест с их статусами |
| **Ошибки** | Что если что-то не так | 404 — сеанс не найден |
| **Правила** | Аутентификация, лимиты, версионирование | Только авторизованные админы меняют сеанс |

**Пример из жизни:** касса кинотеатра и сайт бронирования — **два потребителя одного договора**. Оба должны видеть **одни и те же места** в одном и том же состоянии. Если сайт показывает «свободно», а касса — «занято», значит, **контракт нарушен**.

## 🧠 Анатомия HTTP-запроса

HTTP-запрос — это **текстовое сообщение**, которое клиент отправляет серверу. Оно состоит из **трёх структурных блоков**, разделённых специальными символами:

| Блок | Что содержит | Разделитель |
|------|--------------|-------------|
| **1. Стартовая строка** (Request Line) | Метод + URL (путь и query) + версия HTTP | Завершается `\r\n` |
| **2. Заголовки** (Headers) | Метаданные: хост, авторизация, формат | Каждый заголовок — на своей строке; блок завершается **пустой строкой** |
| **3. Тело** (Body) | Данные запроса (для GET — пустое) | Идёт после пустой строки |

### 📋 Разбор примера построчно

Возьмём запрос на получение списка свободных мест сеанса №42:

```
GET /sessions/42/seats?status=free HTTP/1.1
Host: iskorka.example.com
Authorization: Bearer eyJhbGci...
Accept: application/json

(здесь тело запроса — для GET его нет)
```

Теперь разметим **каждую строку** и покажем, к какому блоку она относится:

| № | Строка в запросе | Блок | Что это значит |
|---|------------------|------|----------------|
| **1** | `GET /sessions/42/seats?status=free HTTP/1.1` | **Стартовая строка** | Вся строка целиком — один блок. Внутри неё три элемента (см. ниже) |
| **2** | `Host: iskorka.example.com` | **Заголовки** | Заголовок `Host` — обязательный в HTTP/1.1, указывает домен |
| **3** | `Authorization: Bearer eyJhbGci...` | **Заголовки** | Заголовок `Authorization` — токен доступа |
| **4** | `Accept: application/json` | **Заголовки** | Заголовок `Accept` — какой формат ответа ожидает клиент |
| **5** | *(пустая строка)* | **Разделитель** | Отделяет заголовки от тела. **Обязательна**, даже если тело пустое |
| **6** | *(тело)* | **Тело** | Для GET — отсутствует |

### 🔍 Что внутри стартовой строки (строка №1)

Стартовая строка — **не три отдельные строки**, а одна, но содержит три элемента:

```
GET   /sessions/42/seats?status=free   HTTP/1.1
└┬─┘  └──────────────┬──────────────┘  └──┬───┘
 │                   │                    │
 │                   │                    └─ Версия HTTP (HTTP/1.1)
 │                   │
 │                   └─ URL: путь + query-параметры
 │                       (/sessions/42/seats — путь;
 │                        status=free — query-параметр)
 │
 └─ HTTP-метод: GET
```

| Элемент стартовой строки | Значение в примере | Что означает |
|--------------------------|--------------------|--------------|
| **Метод** | `GET` | «Хочу получить данные» |
| **URL (путь)** | `/sessions/42/seats` | Ресурс: места сеанса №42 |
| **Query-параметр** | `status=free` | Фильтр: только свободные места |
| **Версия HTTP** | `HTTP/1.1` | Протокол, по которому идёт общение |

### 📊 Сводная схема: где что находится

```
┌──────────────────────────────────────────────────────────┐
│                  HTTP-ЗАПРОС                             │
├──────────────────────────────────────────────────────────┤
│                                                          │
│  ┌─ БЛОК 1: СТАРТОВАЯ СТРОКА ───────────────────────┐    │
│  │  GET /sessions/42/seats?status=free HTTP/1.1     │    │
│  │  └─ метод  └── URL (путь + query) ──┘ └─версия─┘ │    │
│  └──────────────────────────────────────────────────┘    │
│                                                          │
│  ┌─ БЛОК 2: ЗАГОЛОВКИ ──────────────────────────────┐    │
│  │  Host: iskorka.example.com                       │    │
│  │  Authorization: Bearer eyJhbGci...               │    │
│  │  Accept: application/json                        │    │
│  └──────────────────────────────────────────────────┘    │
│                                                          │
│  ┌─ РАЗДЕЛИТЕЛЬ: ПУСТАЯ СТРОКА ─────────────────────┐    │
│  │  (обязательна, даже если тело пустое)            │    │
│  └──────────────────────────────────────────────────┘    │
│                                                          │
│  ┌─ БЛОК 3: ТЕЛО ───────────────────────────────────┐    │
│  │  Для GET — отсутствует                           │    │
│  │  Для POST/PUT/PATCH — содержит данные (JSON)     │    │
│  └──────────────────────────────────────────────────┘    │
│                                                          │
└──────────────────────────────────────────────────────────┘
```

### 💡 Аналогия для запоминания

HTTP-запрос похож на **письмо**:

| Элемент письма | Аналог в HTTP |
|----------------|---------------|
| Строка «Кому: …» с адресом и темой | **Стартовая строка** (метод + URL + версия) |
| Служебные пометки на конверте («Авиапочта», «Срочно») | **Заголовки** |
| Пустая строка после шапки конверта | **Разделитель** |
| Сам текст письма | **Тело** |

### 🆚 Пример запроса с телом (POST)

Для сравнения — запрос на создание брони, где тело **есть**:

```
POST /bookings HTTP/1.1
Host: iskorka.example.com
Authorization: Bearer eyJhbGci...
Content-Type: application/json
Content-Length: 52

{"sessionId": 42, "seatId": 7, "userId": 101}
```

| № | Строка | Блок |
|---|--------|------|
| **1** | `POST /bookings HTTP/1.1` | Стартовая строка (метод POST, URL `/bookings`, версия 1.1) |
| **2** | `Host: iskorka.example.com` | Заголовок |
| **3** | `Authorization: Bearer eyJhbGci...` | Заголовок |
| **4** | `Content-Type: application/json` | Заголовок — **сообщает серверу, что в теле JSON** |
| **5** | `Content-Length: 52` | Заголовок — длина тела в байтах |
| **6** | *(пустая строка)* | Разделитель |
| **7** | `{"sessionId": 42, ...}` | **Тело** — данные брони |

> 💡 **Ключевое различие:** GET не имеет тела, POST/PUT/PATCH — имеют. Заголовки `Content-Type` и `Content-Length` появляются именно тогда, когда тело **есть**.

In [9]:
# ============================================================
#  ЖИВАЯ ДЕМОНСТРАЦИЯ: как выглядит HTTP-запрос и ответ
# ============================================================
# Используем публичный API httpbin.org — он эхом возвращает то,
# что мы ему отправили. Идеально для изучения HTTP.

import requests
import json

url = 'https://httpbin.org/get'
params = {'session_id': '42', 'status': 'free'}
headers = {'Accept': 'application/json'}

print('=' * 70)
print(' ЧТО МЫ ОТПРАВЛЯЕМ:')
print('=' * 70)
print(f'  Метод:  GET')
print(f'  URL:    {url}')
print(f'  Параметры: {params}')
print(f'  Заголовки: {headers}')
print('=' * 70)

try:
    response = requests.get(url, params=params, headers=headers, timeout=10)

    print('\nЧТО ОТВЕТИЛ СЕРВЕР:')
    print('=' * 70)
    print(f'  Код ответа:      {response.status_code}')
    print(f'  Content-Type:    {response.headers.get("Content-Type")}')
    print(f'  Время ответа:    {response.elapsed.total_seconds()*1000:.0f} мс')
    print('=' * 70)
    print('\nТЕЛО ОТВЕТА (JSON):')
    print(json.dumps(response.json(), indent=2, ensure_ascii=False))

except requests.exceptions.RequestException as e:
    print(f'⚠️  Нет соединения с интернетом: {e}')
    print('   Для выполнения упражнения нужен доступ к httpbin.org')

 ЧТО МЫ ОТПРАВЛЯЕМ:
  Метод:  GET
  URL:    https://httpbin.org/get
  Параметры: {'session_id': '42', 'status': 'free'}
  Заголовки: {'Accept': 'application/json'}

ЧТО ОТВЕТИЛ СЕРВЕР:
  Код ответа:      200
  Content-Type:    application/json
  Время ответа:    432 мс

ТЕЛО ОТВЕТА (JSON):
{
  "args": {
    "session_id": "42",
    "status": "free"
  },
  "headers": {
    "Accept": "application/json",
    "Accept-Encoding": "gzip, deflate, zstd",
    "Host": "httpbin.org",
    "User-Agent": "python-requests/2.34.2"
  },
  "method": "GET",
  "origin": "109.248.64.43",
  "url": "https://httpbin.org/get?session_id=42&status=free"
}


---
# 3. JSON и YAML: форматы данных
---

## 🧠 JSON: JavaScript Object Notation

**JSON** создан **Дугласом Крокфордом** в начале 2000-х. Изначально — как подмножество JavaScript, но быстро стал **универсальным форматом обмена данными** благодаря простоте и читаемости.

**Структура JSON:**

| Тип | Пример |
|-----|--------|
| **Объект** | `{"id": 42, "name": "Сеанс"}` |
| **Массив** | `[1, 2, 3]` |
| **Строка** | `"hello"` |
| **Число** | `42`, `3.14` |
| **Логический** | `true`, `false` |
| **null** | `null` |

**Типичные ошибки:**

```json
// ❌ Висящая запятая
{"id": 1, "name": "тест",}

// ❌ Одинарные кавычки
{'id': 1}

// ❌ Комментарии
{"id": 1 // комментарий}
```

**Правильный JSON:**

```json
{
  "sessionId": 42,
  "film": "Король Лев",
  "seats": [
    {"id": 1, "row": 5, "number": 7, "status": "free"},
    {"id": 2, "row": 5, "number": 8, "status": "booked"}
  ]
}
```

## 🧠 YAML: YAML Ain't Markup Language

**YAML** появился в **2001 году** (авторы — Кларк Эванс и Инги Дёт Кэт). YAML **человекочитаемее**, чем JSON, и часто используется для **конфигураций** и **спецификаций API** (OpenAPI, AsyncAPI).

**Пример YAML для той же структуры:**

```yaml
sessionId: 42
film: "Король Лев"
seats:
  - id: 1
    row: 5
    number: 7
    status: free
  - id: 2
    row: 5
    number: 8
    status: booked
```

**Когда что выбирать:**

| Формат | Где используется |
|--------|------------------|
| **JSON** | Передача данных между клиентом и сервером |
| **YAML** | Конфигурации, OpenAPI/AsyncAPI-спецификации, CI/CD |

> 💡 **Правило:** JSON и YAML описывают **одни и те же данные**, но для разных целей. В контрактах API (OpenAPI, AsyncAPI) используется YAML — его удобнее читать и версионировать.

In [ ]:
# ============================================================
#  ПРАКТИКА: парсинг JSON и YAML, поиск ошибок
# ============================================================
import json
import yaml

# --- Верный JSON ---
json_ok = '''
{
  "sessionId": 42,
  "film": "Король Лев",
  "seats": [
    {"id": 1, "status": "free"},
    {"id": 2, "status": "booked"}
  ]
}
'''

parsed = json.loads(json_ok)
print('✅ JSON разобран успешно:')
print(f'  Сеанс: {parsed["sessionId"]}, фильм: {parsed["film"]}')
print(f'  Мест: {len(parsed["seats"])}')
print(f'  Свободных: {sum(1 for s in parsed["seats"] if s["status"] == "free")}')

# --- Ошибочный JSON ---
json_bad = '{"sessionId": 42, "film": "Король Лев",}'

print('\n❌ Попытка разобрать JSON с висящей запятой:')
try:
    json.loads(json_bad)
except json.JSONDecodeError as e:
    print(f'  Ошибка: {e.msg}')
    print(f'  Позиция: строка {e.lineno}, столбец {e.colno}')

# --- YAML ---
yaml_ok = '''
sessionId: 42
film: Король Лев
seats:
  - id: 1
    status: free
  - id: 2
    status: booked
'''

print('\n✅ YAML разобран успешно:')
print(yaml.safe_load(yaml_ok))

print('\n💡 В курсе мы будем писать OpenAPI-спецификацию в YAML,')
print('   а данные передавать в JSON. Понимание обоих форматов — обязательно.')

---
# 4. Контракт: «сначала договориться, потом писать код»
---

## 🧠 Подходы design-first и code-first

| Подход | Суть | Плюсы | Минусы |
|--------|------|-------|--------|
| **Code-first** | Сначала пишем код сервера, потом генерируем документацию | Быстро для прототипов | Контракт «плывёт» за кодом, клиенты ломаются |
| **Design-first** | Сначала пишем **контракт** (OpenAPI/AsyncAPI), потом код | Клиенты и сервер работают по одному договору | Требует дисциплины |

**На этом курсе мы используем design-first.** Причины:

1. **Контракт — это согласие сторон.** Пока клиент и сервер не договорились, кода не должно быть.
2. **Контракт можно проверить** без написания сервера (mock, линтеры).
3. **Контракт переиспользуется** — для документации, генерации кода, mock-серверов, тестов.

## 🎯 Пример из «Искорки»

Представьте, что сайт бронирования и касса кинотеатра делают **одну и ту же бронь**. Если у них **разные** представления о том, как выглядит бронь — они будут конфликтовать:

| Поле | Сайт | Касса |
|------|------|-------|
| Идентификатор | `bookingId` | `id` |
| Дата сеанса | `sessionDate` | `startTime` |
| Статус | `status: "pending"` | `state: 1` |

**Один контракт устраняет эту проблему раз и навсегда.**

> 💡 **Главный вывод модуля:** API-контракт — это **инженерный документ**. Его пишут **до** кода, ревьюят как код, версионируют как код.

---
# 5. Карта курса: где мы находимся
---

## 🗺️ Стили интеграции, которые мы пройдём

```
┌─────────────────────────────────────────────────────────────────┐
│                  КАРТА КУРСА «API»                               │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  Модуль 0 ◄── ВЫ ЗДЕСЬ                                          │
│      │                                                          │
│      ▼                                                          │
│  Модуль 1: Архитектура кейса (C4, домены, интеграционные точки) │
│      │                                                          │
│      ▼                                                          │
│  Модуль 2: Парадигмы интеграции (синхронная / асинхронная /     │
│            событийная)                                          │
│      │                                                          │
│      ▼                                                          │
│  Модуль 3: REST — проектирование                                │
│      │                                                          │
│      ▼                                                          │
│  Модуль 4: OpenAPI, Postman, mock                               │
│      │                                                          │
│      ▼                                                          │
│  Модуль 5: Надёжность и безопасность                            │
│      │                                                          │
│      ├──► Модуль 6: gRPC и GraphQL                              │
│      │                                                          │
│      ├──► Модуль 7: RabbitMQ (очереди)                          │
│      │                                                          │
│      ├──► Модуль 8: Kafka и AsyncAPI (события)                  │
│      │                                                          │
│      ├──► Модуль 9: Согласованность (Outbox, Saga)              │
│      │                                                          │
│      ├──► Модуль 10: Тестирование и CI                          │
│      │                                                          │
│      ▼                                                          │
│  Модуль 11: Итоговый проект                                     │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

## 🎯 Что вы получите в итоге

К концу курса у вас будет:

- **C4-схема** вашего сервиса.
- **OpenAPI-спецификация** (REST).
- **`.proto`-файл** (gRPC).
- **GraphQL-схема**.
- **`asyncapi.yaml`** для событий.
- **Работающая система** с RabbitMQ, Kafka, Postman, CI.

> 💡 **Ключевая идея курса:** одна и та же задача «Искорки» (например, бронирование) будет решаться **разными стилями** — чтобы вы могли **сравнивать** и **выбирать осознанно**.

---
# 6. Практика: настройка окружения и первые запросы
---

## ✅ Чек-лист окружения

Отметьте выполненные пункты:

- [ ] **Python 3.10+** установлен (проверка: `python --version`).
- [ ] **Postman** установлен ([postman.com/downloads](https://www.postman.com/downloads/)).
- [ ] **Docker Desktop** установлен ([docker.com](https://www.docker.com/products/docker-desktop/)).
- [ ] **Редактор кода** (VS Code / PyCharm) с расширением для YAML.
- [ ] **Git** установлен.
- [ ] **Стартовый репозиторий** склонирован (ссылка будет в чате курса).

## 🎯 Практика 1. Отправить 5 запросов и объяснить коды

Ниже — заготовка для отправки запросов к **httpbin.org** (публичный API для учебных целей).

**Задание:** для каждого запроса:
1. Отправьте запрос.
2. Запишите **код ответа**.
3. Объясните своими словами, что этот код означает.

In [ ]:
# ============================================================
#  ПРАКТИКА 1: пять запросов с разными кодами ответа
# ============================================================
import requests

requests_to_make = [
    {'name': '1. Успешный GET',            'method': 'GET',  'url': 'https://httpbin.org/get'},
    {'name': '2. Несуществующий ресурс',   'method': 'GET',  'url': 'https://httpbin.org/status/404'},
    {'name': '3. Ошибка сервера',          'method': 'GET',  'url': 'https://httpbin.org/status/500'},
    {'name': '4. Создание ресурса',        'method': 'POST', 'url': 'https://httpbin.org/post'},
    {'name': '5. Запрещённый запрос',      'method': 'GET',  'url': 'https://httpbin.org/status/403'},
]

code_explanations = {
    200: 'OK — запрос выполнен успешно',
    201: 'Created — ресурс создан',
    204: 'No Content — успех без тела ответа',
    400: 'Bad Request — неверный формат запроса',
    401: 'Unauthorized — нет авторизации',
    403: 'Forbidden — доступ запрещён',
    404: 'Not Found — ресурс не найден',
    409: 'Conflict — конфликт (например, место уже занято)',
    500: 'Internal Server Error — ошибка на сервере',
}

print('=' * 80)
print(' ПРАКТИКА 1: ПЯТЬ ЗАПРОСОВ И ИХ КОДЫ ОТВЕТА')
print('=' * 80)

for req in requests_to_make:
    try:
        if req['method'] == 'GET':
            r = requests.get(req['url'], timeout=10)
        else:
            r = requests.post(req['url'], json={'demo': True}, timeout=10)

        explanation = code_explanations.get(r.status_code, '—')
        print(f"\n{req['name']}")
        print(f"  {req['method']} {req['url']}")
        print(f"  → Код: {r.status_code}")
        print(f"  → Что означает: {explanation}")
    except requests.exceptions.RequestException as e:
        print(f"\n{req['name']}: ⚠️ ошибка соединения — {e}")

print('\n' + '=' * 80)
print(' 📝 Задание: запишите в журнал решений, что вы поняли про коды 200, 404, 500, 201, 403.')

---
# 7. Практика: вопросы к кейсу
---

## 🎯 Практика 2. Чего не хватает в описании кейса?

Прочитайте описание «Искорки» ещё раз. В нём **намеренно** не хватает деталей — это **задел для следующего модуля**.

**Задание:** выпишите **5–7 вопросов**, на которые в кейсе нет ответа. Например:

- Что происходит, если два зрителя одновременно бронируют одно место?
- Сколько времени место удерживается за бронью до оплаты?
- Что делать, если сеанс меняется, а на него уже есть брони?
- Кто такой «администратор» и какие у него права?
- Нужно ли уведомлять зрителей об изменениях?

**Форма для заполнения ниже.** После заполнения обсудите вопросы в парах.

In [ ]:
# ============================================================
#  ПРАКТИКА 2: список вопросов к кейсу
# ============================================================
# Заполните список своими вопросами

open_questions = [
    'Вопрос 1: ...',
    'Вопрос 2: ...',
    'Вопрос 3: ...',
    'Вопрос 4: ...',
    'Вопрос 5: ...',
]

print('=' * 70)
print(' ВОПРОСЫ К КЕЙСУ «ИСКОРКА» (задел для Модуля 1)')
print('=' * 70)
for i, q in enumerate(open_questions, 1):
    print(f'  {i}. {q}')
print('=' * 70)
print('\n💡 Сохраните этот список — на следующем модуле мы начнём')
print('   строить архитектуру, отвечая именно на такие вопросы.')

---
# 8. Журнал решений
---

## 🧠 Зачем вести журнал

В курсе каждый выбор (стиль API, инструмент, формат) должен быть **обоснован**. Журнал решений — это ваш **артефакт для защиты итогового проекта**.

**Структура записи:**

| Поле | Что писать |
|------|------------|
| **Дата** | Когда принято решение |
| **Контекст** | Какая задача стояла |
| **Решение** | Что выбрали |
| **Почему** | Обоснование |
| **От чего отказались** | Альтернативы |

## 📝 Первая запись — по этому модулю

Заполните первую запись в журнале:

| Поле | Ваш ответ |
|------|-----------|
| **Дата** | (сегодня) |
| **Контекст** | Модуль 0. Какой подход к проектированию API выбрать — code-first или design-first? |
| **Решение** | — |
| **Почему** | — |
| **От чего отказались** | — |

> 💡 **Правило:** в журнале **не бывает неправильных** решений. Бывает **необоснованных**. Обоснование — важнее выбора.

---
# 9. Сквозной словарь терминов
---

| Термин | Определение |
|--------|-------------|
| **API** | Контракт между потребителем и поставщиком данных/функций |
| **Контракт** | Формальное описание того, как клиент и сервер взаимодействуют |
| **HTTP** | Протокол передачи данных, лежащий в основе веба |
| **HTTP-запрос** | Сообщение от клиента серверу (метод + URL + заголовки + тело) |
| **HTTP-ответ** | Сообщение от сервера клиенту (код + заголовки + тело) |
| **JSON** | Формат обмена данными, основанный на JavaScript-объектах |
| **YAML** | Человекочитаемый формат для конфигураций и спецификаций |
| **Код ответа** | Число в HTTP-ответе (200, 404, 500…), показывающее результат |
| **Design-first** | Подход, при котором контракт пишется до кода |
| **Code-first** | Подход, при котором код пишется до контракта |

> 💡 Словарь пополняется в каждом модуле. К концу курса — это **глоссарий** для защиты проекта.

---
# 10. Контрольная точка: квиз по HTTP
---

## 📝 Квиз (10 минут)

**Задание:** для каждой ситуации выберите правильный код ответа.

| № | Ситуация | Код |
|---|----------|-----|
| 1 | Зритель успешно получил список мест | ? |
| 2 | Зритель отправил запрос на бронь, но не указал id сеанса | ? |
| 3 | Зритель пытается отменить бронь, которая ему не принадлежит | ? |
| 4 | Администратор изменил время сеанса — успешно | ? |
| 5 | Клиент пытается забронировать место, которое только что заняли | ? |
| 6 | Запрос на несуществующий сеанс | ? |
| 7 | Внутренняя ошибка сервера при обработке брони | ? |
| 8 | Бронь успешно создана | ? |
| 9 | Слишком много запросов от одного клиента | ? |
| 10 | Бронь успешно отменена (без тела ответа) | ? |

**Коды для выбора:** `200`, `201`, `204`, `400`, `401`, `403`, `404`, `409`, `429`, `500`.

In [ ]:
# ============================================================
#  КВИЗ: проверьте себя
# ============================================================

quiz = [
    {'situation': 'Зритель успешно получил список мест', 'answer': 200},
    {'situation': 'Запрос на бронь без id сеанса', 'answer': 400},
    {'situation': 'Зритель отменяет чужую бронь', 'answer': 403},
    {'situation': 'Администратор успешно изменил сеанс', 'answer': 200},
    {'situation': 'Место уже занято', 'answer': 409},
    {'situation': 'Несуществующий сеанс', 'answer': 404},
    {'situation': 'Внутренняя ошибка сервера', 'answer': 500},
    {'situation': 'Бронь создана', 'answer': 201},
    {'situation': 'Слишком много запросов', 'answer': 429},
    {'situation': 'Бронь отменена (без тела)', 'answer': 204},
]

explanations = {
    200: 'OK — успех',
    201: 'Created — ресурс создан',
    204: 'No Content — успех без тела',
    400: 'Bad Request — неверный формат запроса',
    401: 'Unauthorized — нет авторизации',
    403: 'Forbidden — доступ запрещён',
    404: 'Not Found — не найдено',
    409: 'Conflict — конфликт',
    429: 'Too Many Requests — превышен лимит',
    500: 'Internal Server Error — ошибка сервера',
}

print('=' * 90)
print(' ОТВЕТЫ К КВИЗУ')
print('=' * 90)
for i, q in enumerate(quiz, 1):
    code = q['answer']
    print(f'  {i:>2}. {q["situation"]:<55} → {code} ({explanations[code]})')
print('=' * 90)

print('\n💡 Если вы ошиблись больше чем в 3 вопросах — повторите разделы 2 и 3.')

---
# 📋 Итоги модуля
---

## ✅ Что вы должны были освоить

- [ ] Объяснить, что такое API и зачем нужен контракт.
- [ ] Прочитать HTTP-запрос и HTTP-ответ.
- [ ] Различать JSON и YAML и понимать, где что используется.
- [ ] Отличать design-first от code-first.
- [ ] Понимать карту курса и место каждого модуля.
- [ ] Отправить запрос к публичному API и разобрать JSON-ответ.
- [ ] Знать назначение кодов 200, 201, 204, 400, 403, 404, 409, 500.

## 📦 Артефакты модуля

1. **Рабочее окружение** — Python, Postman, Docker, редактор, Git.
2. **Заметки по первым запросам** — 5 запросов с объяснением кодов.
3. **Список вопросов к кейсу** — 5–7 открытых вопросов для Модуля 1.
4. **Первая запись в журнале решений** — почему выбран design-first.

## 🏠 Домашнее задание

1. **Отправить 3 запроса к `jsonplaceholder.typicode.com`**:
   - `GET /posts` — получить список постов.
   - `GET /posts/1` — получить один пост.
   - `POST /posts` — создать новый пост.

   Для каждого — записать код ответа и объяснить.

2. **Найти в описании кейса «Искорка» хотя бы 5 неоднозначностей** и записать, как бы вы их уточнили у заказчика.

3. **Прочитать главу 1 книги «API Design Patterns» (J. J. Geewax)** — о том, что такое API как контракт.

## 🔮 Что дальше

**Модуль 1. Архитектурный контекст кейса.** На следующем занятии мы:

- Построим **C4-модель** для «Искорки».
- Проведём мини-**Event Storming** по четырём сценариям.
- Сформулируем **5–7 нефункциональных требований** (НФТ).
- Заполним **таблицу интеграционных точек** — «красную нить» всего курса.

> 💡 **Совет:** перечитайте описание кейса перед следующим занятием. Ваши вопросы из практики 2 станут основой для обсуждения.

---
## 📚 Источники модуля

- **Fielding, R. (2000).** *Architectural Styles and the Design of Network-based Software Architectures* — диссертация, где сформулирован REST.
- **Berners-Lee, T. (1991).** *The Original HTTP Document* — первое описание HTTP.
- **Crockford, D. (2002).** *JSON.org* — спецификация JSON.
- **IETF RFC 9110.** *HTTP Semantics* — официальная спецификация HTTP (2022).
- **Moeser, J. (2019).** *OpenAPI Specification 3.1* — спецификация для API-контрактов.
- **Geewax, J. J. (2021).** *API Design Patterns* — книга о проектировании API как контракта.
- **Postman Learning Center.** [learning.postman.com](https://learning.postman.com/) — документация по Postman.

---

*Курс «Проектирование и разработка API». Модуль 0. 2026.*